In [1]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from itertools import islice
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# Helper Functions

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

SRC_DIR = Path.cwd().parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from eda_helpers import (
    DEFAULT_PERCENTILES,
    build_distribution_summary,
    build_outlier_summary,
    build_segment_summary,
    cast_numeric_fields,
    plot_bucket_counts,
    plot_correlation_heatmap,
    plot_full_and_clipped_boxplot,
    plot_histogram_with_log,
    plot_scatter_pairs,
    plot_slices_of_segments_boxplot,
)

CHART_DIR = Path("charts")

# Statistic summary - Priced in April

In [3]:
PROJECT_ID = "gannett-datascience"
TABLE_ID = "gannett-datascience.test_results_zone.ss_test_result_v3-2"

NUMERIC_FIELDS = [
    "frequency",
    "breadth",
    "tenure",
    "tt_cost"
]
SEGMENT_FIELDS = [
    "src_risk_tier",
    "cohort",
    "Treatment",
    "contact_channel",
    "status",
    "contact_timing",
    "repeatedly_called"
]
IDS = [
    "billing_account",
    "id_subscrip",
    "email_date"
]

client = bigquery.Client(project=PROJECT_ID)

In [4]:
query = f"""
SELECT
  {", ".join(NUMERIC_FIELDS + SEGMENT_FIELDS + IDS)}
FROM `{TABLE_ID}`
where email_date < '2026-05-01'
"""
df = client.query(query).to_dataframe()
df = cast_numeric_fields(df, NUMERIC_FIELDS)
df['contact_channels'] = df['contact_channel'].replace({'Online first': 'Contacted both ways', 'Called-In first': 'Contacted both ways'})
df['src_risk_tier'] = df['src_risk_tier']+' risk'
df_action = df[df["status"].ne("No Action yet")].copy()
df_no_action = df[df["status"].eq("No Action yet")].copy()
df_save = df[df["status"].eq("saved")].copy()
df_stop = df[df["status"].eq("stoped")].copy()

In [5]:
print("Shapes: total ", df.shape, "action ", df_action.shape, "no action ", df_no_action.shape, "save ", df_save.shape, "stop ", df_stop.shape)
# df_action.info()

Shapes: total  (9398, 15) action  (1531, 15) no action  (7867, 15) save  (589, 15) stop  (942, 15)


In [6]:
# df_action.describe(include="all").T

In [7]:
percentiles = DEFAULT_PERCENTILES
dfs = {"No Action": df_no_action, "Action": df_action, "Saved": df_save, "Stopped": df_stop}

for name, df_i in dfs.items():
    print(f"Distribution Summary for {name} users:")
    distribution_summary = build_distribution_summary(df_i, NUMERIC_FIELDS, percentiles)
    display(distribution_summary)

Distribution Summary for No Action users:


,field,row_count,null_count,null_pct,zero_count,negative_count,min,max,mean,median,std,p01,p05,p10,p25,p50,p75,p90,p95,p99
0,frequency,7867,6,0.08,1632,0,0.00,91.00,38.46,29.00,35.37,0.00,0.00,0.00,2.00,29.00,77.00,90.00,91.00,91.00
1,breadth,7867,6,0.08,2466,0,0.00,21.00,4.18,3.00,4.24,0.00,0.00,0.00,0.00,3.00,7.00,10.00,12.00,16.00
2,tenure,7867,6,0.08,0,0,208.00,"27,867.00","1,382.95","1,354.00",886.36,324.00,337.00,504.00,997.00,"1,354.00","1,635.00","2,069.00","2,460.00","3,405.80"
3,tt_cost,7867,6,0.08,32,0,0.00,"9,279.71",571.25,502.83,439.16,107.89,112.89,166.27,374.25,502.83,704.96,949.38,"1,279.15","1,764.45"


Distribution Summary for Action users:


,field,row_count,null_count,null_pct,zero_count,negative_count,min,max,mean,median,std,p01,p05,p10,p25,p50,p75,p90,p95,p99
0,frequency,1531,3,0.20,183,0,0.00,91.00,45.90,46.00,34.53,0.00,0.00,0.00,9.00,46.00,82.00,90.00,91.00,91.00
1,breadth,1531,3,0.20,292,0,0.00,20.00,5.04,4.00,4.19,0.00,0.00,0.00,1.00,4.00,8.00,11.00,13.00,15.73
2,tenure,1531,3,0.20,0,0,323.00,"11,789.00","1,501.89","1,450.00",820.37,325.27,358.35,689.70,"1,144.00","1,450.00","1,728.25","2,332.00","2,639.65","3,790.92"
3,tt_cost,1531,3,0.20,0,0,25.91,"3,925.74",504.38,462.87,336.87,108.23,119.76,169.16,289.29,462.87,621.14,826.45,"1,005.34","1,788.55"


Distribution Summary for Saved users:


,field,row_count,null_count,null_pct,zero_count,negative_count,min,max,mean,median,std,p01,p05,p10,p25,p50,p75,p90,p95,p99
0,frequency,589,2,0.34,28,0,0.00,91.00,56.41,67.00,32.41,0.00,1.00,5.60,26.00,67.00,88.00,91.00,91.00,91.00
1,breadth,589,2,0.34,64,0,0.00,18.00,5.69,5.00,4.06,0.00,0.00,0.00,2.00,5.00,9.00,11.00,13.00,16.00
2,tenure,589,2,0.34,0,0,323.00,"10,500.00","1,542.57","1,454.00",832.41,324.86,346.00,724.00,"1,148.00","1,454.00","1,777.50","2,416.40","2,704.80","3,722.36"
3,tt_cost,589,2,0.34,0,0,36.46,"3,496.50",468.53,428.24,309.50,107.85,114.09,163.43,268.02,428.24,572.52,788.32,949.16,"1,680.29"


Distribution Summary for Stopped users:


,field,row_count,null_count,null_pct,zero_count,negative_count,min,max,mean,median,std,p01,p05,p10,p25,p50,p75,p90,p95,p99
0,frequency,942,1,0.11,155,0,0.00,91.00,39.35,33.00,34.21,0.00,0.00,0.00,4.00,33.00,76.00,89.00,91.00,91.00
1,breadth,942,1,0.11,228,0,0.00,20.00,4.64,4.00,4.22,0.00,0.00,0.00,1.00,4.00,8.00,11.00,13.00,15.00
2,tenure,942,1,0.11,0,0,324.00,"11,789.00","1,476.51","1,421.00",812.18,329.80,390.00,510.00,"1,142.00","1,421.00","1,696.00","2,245.00","2,588.00","3,752.00"
3,tt_cost,942,1,0.11,0,0,25.91,"3,925.74",526.75,483.85,351.18,109.82,154.86,169.50,311.02,483.85,645.02,839.16,"1,050.46","1,824.59"


In [8]:
for segment in SEGMENT_FIELDS:
    for name, df_i in dfs.items():
        print(f"\nSegment summary by {segment} for {name} users:")
        segment_summary = build_segment_summary(df_i, segment, NUMERIC_FIELDS)
        display(segment_summary)


Segment summary by src_risk_tier for No Action users:


,src_risk_tier,users,avg_frequency,median_frequency,p90_frequency,avg_breadth,median_breadth,p90_breadth,avg_tenure,median_tenure,p90_tenure,avg_tt_cost,median_tt_cost,p90_tt_cost
0,1. Low risk,1901,71.93,84.00,91.00,6.70,7.00,12.00,"1,565.33","1,502.00","2,393.00",621.90,523.48,"1,151.23"
2,3. Medium risk,1267,10.08,3.00,33.00,1.70,0.00,6.00,"1,278.57","1,305.00","1,670.40",597.58,568.24,888.22
1,2. Med-Low risk,1065,40.28,41.00,71.60,5.03,5.00,10.00,"1,469.87","1,425.00","2,420.80",693.48,607.12,"1,282.92"
3,4. Med-High risk,375,6.44,3.00,18.00,1.41,0.00,4.60,879.26,840.00,"1,090.00",394.05,388.48,534.33
4,5. High risk,32,12.84,1.00,81.70,2.69,1.00,9.80,808.41,724.50,"1,335.50",266.31,206.63,613.99



Segment summary by src_risk_tier for Action users:


,src_risk_tier,users,avg_frequency,median_frequency,p90_frequency,avg_breadth,median_breadth,p90_breadth,avg_tenure,median_tenure,p90_tenure,avg_tt_cost,median_tt_cost,p90_tt_cost
0,1. Low risk,560,69.38,82.00,91.00,6.74,7.00,12.00,"1,632.72","1,555.50","2,404.20",508.92,433.07,848.12
1,2. Med-Low risk,199,38.49,38.00,71.00,4.97,4.00,10.00,"1,787.80","1,725.00","2,515.20",669.61,624.04,"1,111.61"
2,3. Medium risk,170,12.21,6.00,36.20,2.11,1.00,7.00,"1,300.71","1,388.00","1,663.00",500.34,484.44,800.63
3,4. Med-High risk,74,10.16,4.50,27.10,2.23,1.00,6.00,"1,027.09",966.00,"1,481.70",336.22,303.86,500.42
4,5. High risk,1,1.00,1.00,1.00,1.00,1.00,1.00,"1,241.00","1,241.00","1,241.00",413.25,413.25,413.25



Segment summary by src_risk_tier for Saved users:


...

,repeatedly_called,users,avg_frequency,median_frequency,p90_frequency,avg_breadth,median_breadth,p90_breadth,avg_tenure,median_tenure,p90_tenure,avg_tt_cost,median_tt_cost,p90_tt_cost
0,0,1518,45.69,45.00,90.00,5.03,4.00,11.00,"1,500.95","1,450.00","2,328.00",505.15,463.20,826.59
1,1,13,73.00,89.00,91.00,6.00,5.00,9.90,"1,619.92","1,432.00","2,777.30",407.48,418.10,701.71



Segment summary by repeatedly_called for Saved users:


,repeatedly_called,users,avg_frequency,median_frequency,p90_frequency,avg_breadth,median_breadth,p90_breadth,avg_tenure,median_tenure,p90_tenure,avg_tt_cost,median_tt_cost,p90_tt_cost
0,0,576,56.06,66.00,91.00,5.68,5.00,11.00,"1,540.95","1,454.00","2,394.00",469.80,430.82,788.54
1,1,13,73.00,89.00,91.00,6.00,5.00,9.90,"1,619.92","1,432.00","2,777.30",407.48,418.10,701.71



Segment summary by repeatedly_called for Stopped users:


,repeatedly_called,users,avg_frequency,median_frequency,p90_frequency,avg_breadth,median_breadth,p90_breadth,avg_tenure,median_tenure,p90_tenure,avg_tt_cost,median_tt_cost,p90_tt_cost
0,0,942,39.35,33.00,89.00,4.64,4.00,11.00,"1,476.51","1,421.00","2,245.00",526.75,483.85,839.16


# stoped vs saved - Histogram

In [10]:
for col in NUMERIC_FIELDS:
    for name, df_i in islice(reversed(dfs.items()), 2):
        plot_histogram_with_log(
            data=df_i,
            metric=col,
            group=name,
            chart_folder=str(CHART_DIR / "histograms" / name),
        )

# stoped vs saved - Correlation, Pairplot, Scatterplot

In [11]:
for name, df_i in islice(reversed(dfs.items()), 2):
    plot_correlation_heatmap(
        df_i,
        NUMERIC_FIELDS,
        name,
        chart_folder=str(CHART_DIR / "correlations"),
    )

In [12]:
# sample_df = df[NUMERIC_FIELDS].dropna().sample(
#     min(5000, len(df.dropna())),
#     random_state=42
# )

# sns.pairplot(sample_df)
# plt.show()

In [13]:
pairs = [
    ("frequency", "breadth"),
    ("frequency", "tenure"),
    ("frequency", "tt_cost"),
    ("breadth", "tenure"),
    ("breadth", "tt_cost"),
    ("tenure", "tt_cost")
]

for name, df_i in islice(reversed(dfs.items()), 2):
    plot_scatter_pairs(
        df_i,
        pairs,
        chart_folder=str(CHART_DIR / "scatter" / name),
        file_name="{pair}.png",
    )

# stoped vs saved - Custom bucket

In [14]:
for name, df_i in islice(reversed(dfs.items()), 2):
    df_i["tenure_bucket"] = pd.cut(
        df_i["tenure"],
        bins=[-np.inf, 0, 30, 90, 180, 365, 730, np.inf],
        labels=[
            "0 or less",
            "1-30 days",
            "31-90 days",
            "91-180 days",
            "181-365 days",
            "1-2 years",
            "2+ years"
        ]
    )
    bucket_cols = ["tenure_bucket"] #, "cost_bucket", "frequency_bucket"]
    for col in bucket_cols:
        bucket_counts = plot_bucket_counts(
            df_i,
            col,
            chart_folder=str(CHART_DIR / "bucket_counts"),
            file_name=f"{col}_counts_{name}_users.png"
        )
        display(bucket_counts)

,tenure_bucket,count
0,2+ years,822
1,1-2 years,78
2,181-365 days,41
3,NaN,1
4,0 or less,0
5,1-30 days,0
6,31-90 days,0
7,91-180 days,0


,tenure_bucket,count
0,2+ years,526
1,181-365 days,37
2,1-2 years,24
3,NaN,2
4,0 or less,0
5,1-30 days,0
6,31-90 days,0
7,91-180 days,0


# Business Boxplots

In [18]:
# No Action vs Had Action
df["action_group"] = np.where(
    df["status"].eq("No Action yet"),
    "No Action yet",
    "Had Action"
)
_ = plot_full_and_clipped_boxplot(
    data=df,
    metrics=NUMERIC_FIELDS,
    group_col="action_group",
    group_order=["No Action yet", "Had Action"],
    chart_title="Metrics: No Action vs Had Action",
    figsize=(10, 5),
    chart_folder=str(CHART_DIR / "boxplots"),
    file_name="metrics_no_action_vs_had_action.png",
)

In [19]:
# Stoped vs Saved
# Repeated called vs Not
df_action["repeated_call_group"] = np.where(
    df_action["repeatedly_called"].fillna(0).astype(int).eq(1),
    "Repeatedly Called",
    "Called Once"
)
group_kwargs = {
    "group_col": ["status", "repeated_call_group"],
    "group_order": [["saved", "stoped"], ["Repeatedly Called", "Called Once"]]
}

for col, order in zip(group_kwargs["group_col"], group_kwargs["group_order"]):
    _ = plot_full_and_clipped_boxplot(
        data=df_action,
        metrics=NUMERIC_FIELDS,
        group_col=col,
        group_order=order,
        chart_title=f"Metrics: {order[0]} vs {order[1]}",
        figsize=(10, 5),
        chart_folder=str(CHART_DIR / "boxplots"),
        file_name=f"metrics_{order[0]}_vs_{order[1]}.png",
    )

In [14]:
# Stoped vs Saved outliers
for name, df_i in islice(reversed(dfs.items()), 2):
    outlier_df = build_outlier_summary(df_i, NUMERIC_FIELDS)
    print(f"Outlier Summary for {name} users:")
    display(outlier_df)

Outlier Summary for Stopped users:


,field,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,frequency,4.00,76.00,72.00,-104.00,184.00,0,0.00
1,breadth,1.00,8.00,7.00,-9.50,18.50,1,0.11
2,tenure,"1,142.00","1,696.00",554.00,311.00,"2,527.00",53,5.63
3,tt_cost,311.02,645.02,334.00,-189.98,"1,146.02",39,4.14


Outlier Summary for Saved users:


,field,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,frequency,26.00,88.00,62.00,-67.00,181.00,0,0.00
1,breadth,2.00,9.00,7.00,-8.50,19.50,0,0.00
2,tenure,"1,148.00","1,777.50",629.50,203.75,"2,721.75",28,4.75
3,tt_cost,268.02,572.52,304.50,-188.72,"1,029.27",21,3.57


In [21]:
# Slices of Segments boxplots
slice_fields=["contact_channels", "cohort", "src_risk_tier", "contact_timing"]
res = plot_slices_of_segments_boxplot(
    df_action,
    metrics=NUMERIC_FIELDS,
    slice_fields=slice_fields,
    chart_folder=str(CHART_DIR / "boxplots"),
    full_file_name="segment_slices_full.png",
    clipped_file_name="segment_slices_clipped.png",
    standardized_clipped_file_name="segment_slices_standardized_clipped.png",
    slices_per_file=6,
    n_cols=2
)

,contact_channels,cohort,src_risk_tier,contact_timing,Treatment,users
0,Called-In Cancel Flow,Three-Offer Cohort,1. Low risk,Contact Before Pricing,Control,87
1,Called-In Cancel Flow,Three-Offer Cohort,1. Low risk,Contact Before Pricing,Midpoint,84
2,Called-In Cancel Flow,Three-Offer Cohort,1. Low risk,Contact Before Pricing,Tiered,75
3,Called-In Cancel Flow,Three-Offer Cohort,1. Low risk,Contact On/After Pricing,Control,41
4,Called-In Cancel Flow,Three-Offer Cohort,1. Low risk,Contact On/After Pricing,Midpoint,50


Preparing 29 unique segment combinations...
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.
No groups meet the minimum sample size.


# WIP

In [ ]:
# distribution_summary.to_csv("distribution_summary.csv")
# quality_df.to_csv("data_quality_summary.csv", index=False)
# outlier_df.to_csv("outlier_summary.csv", index=False)

# print("Saved summary files.")

In [ ]:
# percentile_query = f"""
# SELECT
#   COUNT(*) AS row_count,

#   APPROX_QUANTILES(tenure, 100)[OFFSET(1)] AS tenure_p01,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(25)] AS tenure_p25,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(50)] AS tenure_p50,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(75)] AS tenure_p75,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(90)] AS tenure_p90,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(95)] AS tenure_p95,
#   APPROX_QUANTILES(tenure, 100)[OFFSET(99)] AS tenure_p99,

#   APPROX_QUANTILES(total_cost, 100)[OFFSET(1)] AS total_cost_p01,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(25)] AS total_cost_p25,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(50)] AS total_cost_p50,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(75)] AS total_cost_p75,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(90)] AS total_cost_p90,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(95)] AS total_cost_p95,
#   APPROX_QUANTILES(total_cost, 100)[OFFSET(99)] AS total_cost_p99,

#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(1)] AS pages_p01,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(25)] AS pages_p25,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(50)] AS pages_p50,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(75)] AS pages_p75,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(90)] AS pages_p90,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(95)] AS pages_p95,
#   APPROX_QUANTILES(viewed_number_of_pages, 100)[OFFSET(99)] AS pages_p99

# FROM `{TABLE_ID}`
# """

# percentiles_bq = client.query(percentile_query).to_dataframe()
# percentiles_bq